# DATA 620 — Project 1
#### **Course:** DATA 620 — Web Analytics
#### **Assignment:** Project 1


**Group Members:**

Crystal Quezada

Nana Kwasi Danquah

Muhammad Suffyan Khan

# Objective

The objective of the first project is to:

1. Identify and load a network dataset that has some categorical information available for each node.
2. For each of the nodes in the dataset, calculate degree centrality and eigenvector centrality.
3. Compare your centrality measures across your categorical groups.

# Video Presentation

**Video Link:**

# 1. Setup and Imports

Here we import all of the libraries we'll be using.

In [1]:
import pandas as pd
import networkx as nx
import ast

# 2. Load Graph Dataset

After loading the data into dataframes, we add the collaboration data into a network graph, and then we calculate degree centrality and eigenvector centrality.

In [2]:
edges = pd.read_csv("https://raw.githubusercontent.com/crystaliquezada/data620_week2_3/refs/heads/main/edges.csv")
nodes = pd.read_csv("https://raw.githubusercontent.com/crystaliquezada/data620_week2_3/refs/heads/main/nodes.csv")

G = nx.from_pandas_edgelist(
    edges,
    source="id_0",
    target="id_1"
)

degree_cent = nx.degree_centrality(G)
eigen_cent = nx.eigenvector_centrality(G, max_iter=1000)

print(nodes.columns)
nodes.head()

Index(['spotify_id', 'name', 'followers', 'popularity', 'genres',
       'chart_hits'],
      dtype='object')


,spotify_id,name,followers,popularity,genres,chart_hits
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,"['nordic house', 'russelater']",['no (3)']
1,4lDiJcOJ2GLCK6p9q5BgfK,Kontra K,1999676.0,72,"['christlicher rap', 'german hip hop']","['at (44)', 'de (111)', 'lu (22)', 'ch (31)', ..."
2,652XIvIBNGg3C0KIGEJWit,Maxim,34596.0,36,[],['de (1)']
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,"['dancehall', 'lovers rock', 'modern reggae', ...","['at (1)', 'de (1)']"
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,"['classic swedish pop', 'norrbotten indie', 's...",['se (6)']


# 3. Clean the Data

The following lines of code clean the artist data. We want to ensure that the dataset contains only valid network participants across different music genres. Artists that do not appear in the collaboration network are removed, and the genre data is converted from text into a usable list. A primary genre is then assigned to each artist, and artists with missing genre information are dropped. The code also maps degree centrality and eigenvector centrality values to each artist, removes records with missing centrality measures, and filters out genres with fewer than 10 artists.

In [3]:
nodes = nodes[nodes["spotify_id"].isin(G.nodes())]

def parse_genres(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

nodes["genres"] = nodes["genres"].apply(parse_genres)

nodes["primary_genre"] = nodes["genres"].apply(
    lambda x: x[0] if len(x) > 0 else None
)

nodes = nodes.dropna(subset=["primary_genre"])


In [4]:
nodes["degree_centrality"] = nodes["spotify_id"].map(degree_cent)
nodes["eigenvector_centrality"] = nodes["spotify_id"].map(eigen_cent)

nodes = nodes.dropna(subset=["degree_centrality", "eigenvector_centrality"])

genre_counts = nodes["primary_genre"].value_counts()
valid_genres = genre_counts[genre_counts >= 10].index

nodes = nodes[nodes["primary_genre"].isin(valid_genres)]
nodes.head()



,spotify_id,name,followers,popularity,genres,chart_hits,primary_genre,degree_centrality,eigenvector_centrality
0,48WvrUGoijadXXCsGocwM4,Byklubben,1738.0,24,"[nordic house, russelater]",['no (3)'],nordic house,0.000013,5.831591e-09
3,3dXC1YPbnQPsfHPVkm1ipj,Christopher Martin,249233.0,52,"[dancehall, lovers rock, modern reggae, reggae...","['at (1)', 'de (1)']",dancehall,0.000254,1.794193e-03
4,74terC9ol9zMo8rfzhSOiG,Jakob Hellman,21193.0,39,"[classic swedish pop, norrbotten indie, swedis...",['se (6)'],classic swedish pop,0.000013,5.086154e-08
6,71BhXa24Zf5zcikUb00l2N,Juice,11312.0,37,"[swedish drill, swedish hip hop, swedish trap,...",['se (4)'],swedish drill,0.000026,2.295402e-06
7,3TG1RXLaEhHz5SIPMWahit,Nehuda,36252.0,31,[francoton],['fr (1)'],francoton,0.000026,1.185167e-04
